# Task 3 - Echocardiogram video processing
I read an ultrasound video frame by frame, enhance each frame and show it next to the original.

Press **q** to close the window. If you have no display, set `HEADLESS = True`.

In [1]:
import cv2
import numpy as np
import os

HEADLESS = os.environ.get('HEADLESS', '0') == '1'
GAMMA = 0.6

# gamma lookup table, made once so it is fast for every frame
table = np.array([255 * (i/255)**GAMMA for i in range(256)]).astype(np.uint8)

### Enhancement for one frame
Order: equalize -> JET -> color balance -> log -> gamma.

In [2]:
def enhance(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    eq = cv2.equalizeHist(gray)                  # fix murky contrast
    heat = cv2.applyColorMap(eq, cv2.COLORMAP_JET)  # heatmap

    # color balance (same average for B, G, R)
    b, g, r = cv2.split(heat.astype(np.float32))
    avg = (b.mean() + g.mean() + r.mean()) / 3
    bal = cv2.merge([b*avg/b.mean(), g*avg/g.mean(), r*avg/r.mean()])
    bal = np.clip(bal, 0, 255).astype(np.uint8)

    # log transform to show dark chambers
    c = 255 / np.log(1 + bal.max())
    lg = (c * np.log(1 + bal.astype(np.float32))).astype(np.uint8)

    # gamma to reduce the bright white noise
    return cv2.LUT(lg, table)

### Video loop

In [3]:
cap = cv2.VideoCapture('data/echo_sample.mp4')
fps = cap.get(cv2.CAP_PROP_FPS) or 25
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

out_video = cv2.VideoWriter('output/echo_side_by_side.mp4', cv2.VideoWriter_fourcc(*'mp4v'), fps, (w*2, h))

count = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break          # video finished

    result = enhance(frame)
    left = frame.copy()
    right = result.copy()
    cv2.putText(left, 'RAW', (5, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,255), 2)
    cv2.putText(right, 'ENHANCED', (5, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,255), 2)

    both = np.hstack([left, right])
    out_video.write(both)

    if count in (0, 30, 60):
        cv2.imwrite(f'output/screenshot_{count}.png', both)

    if not HEADLESS:
        cv2.imshow('raw vs enhanced', both)
        if cv2.waitKey(int(1000/fps)) & 0xFF == ord('q'):
            break
    count += 1

cap.release()
out_video.release()
cv2.destroyAllWindows() if not HEADLESS else None
print('frames done:', count)

/tmp/ipykernel_149/1044065213.py:13: RuntimeWarning: overflow encountered in scalar add
  c = 255 / np.log(1 + bal.max())
/tmp/ipykernel_149/1044065213.py:13: RuntimeWarning: divide by zero encountered in log
  c = 255 / np.log(1 + bal.max())


frames done: 75


### Conclusion
(write in your own words how the enhanced video looks compared to the raw one)